# 12 — Four-signal RHI fusion (completes Experiment 5)

Notebook 09 fused three signals and the result (AUC 0.622) did **not** beat the
survival signal alone (0.753). The stated hypothesis was that the missing
fourth signal — distress language — carried the complementary information.
Notebook 11 now supplies it, so this notebook tests that hypothesis directly.

This is a real test with a real chance of failing. If the four-signal fusion
still does not beat survival alone, that is the finding and it gets reported
as such, and the hypothesis from notebook 09 is refuted rather than quietly
dropped.

Reads: `survival_scored.csv`, `cashflow_projections.csv`, `spend_anomalies.csv`,
`distress_scores.csv`
Writes: `reports/experiment5_fusion_full.csv`, `reports/table2_ablation_4signal.csv`

In [1]:
import pandas as pd, numpy as np, os
from itertools import combinations
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import StratifiedKFold, cross_val_score

PROCESSED="../data/processed"; REPORTS="../reports"
os.makedirs(REPORTS, exist_ok=True)

survival  = pd.read_csv(f"{PROCESSED}/survival_scored.csv")
cashflow  = pd.read_csv(f"{PROCESSED}/cashflow_projections.csv")
anomalies = pd.read_csv(f"{PROCESSED}/spend_anomalies.csv")
distress  = pd.read_csv(f"{PROCESSED}/distress_scores.csv")

anom = anomalies.groupby("object_id")["is_anomaly"].mean().rename("anomaly_rate").reset_index()

m = (survival.merge(cashflow, left_on="id", right_on="object_id", how="inner")
              .merge(anom, on="object_id", how="left")
              .merge(distress, on="object_id", how="left"))
m["anomaly_rate"]   = m["anomaly_rate"].fillna(0)
m["cashflow_urgency"] = 1/(1 + m["months_to_zero_projected"].clip(lower=0)/12)

n_before = len(m)
m = m.dropna(subset=["distress_score"])
print(f"[merge] {n_before:,} companies had the first three signals")
print(f"[merge] {len(m):,} of those also carry a distress score (text available)")
print(f"[merge] {n_before-len(m):,} dropped for missing text -- report this coverage figure")

[merge] 1,898 companies had the first three signals
[merge] 1,892 of those also carry a distress score (text available)
[merge] 6 dropped for missing text -- report this coverage figure


## Full ablation over all four signals (2⁴ − 1 = 15 subsets)

In [2]:
SIGNALS = ["p_exhaust_6m","cashflow_urgency","anomaly_rate","distress_score"]
LABEL   = {"p_exhaust_6m":"survival","cashflow_urgency":"cash-flow",
           "anomaly_rate":"anomaly","distress_score":"distress-language"}
y = m["event"]
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=7) if y.value_counts().min()>=5 else 5

rows=[]
for r in range(1, len(SIGNALS)+1):
    for combo in combinations(SIGNALS, r):
        s = cross_val_score(LogisticRegression(max_iter=1000),
                            m[list(combo)].fillna(0), y, cv=cv, scoring="roc_auc")
        rows.append({"signals": " + ".join(LABEL[c] for c in combo),
                     "n_signals": r, "auc_mean": s.mean(), "auc_std": s.std()})

ab = pd.DataFrame(rows).sort_values("auc_mean", ascending=False).reset_index(drop=True)
ab.to_csv(f"{REPORTS}/table2_ablation_4signal.csv", index=False)
print(ab.to_string(index=False))

                                           signals  n_signals  auc_mean  auc_std
                                          survival          1  0.753492 0.027151
                                survival + anomaly          2  0.739195 0.019046
                                           anomaly          1  0.636357 0.036862
                    survival + cash-flow + anomaly          3  0.616215 0.033134
                               cash-flow + anomaly          2  0.597013 0.030270
            survival + anomaly + distress-language          3  0.554492 0.058384
                       anomaly + distress-language          2  0.554344 0.058286
                              survival + cash-flow          2  0.552957 0.022607
                      survival + distress-language          2  0.547103 0.053525
                                 distress-language          1  0.546944 0.053484
survival + cash-flow + anomaly + distress-language          4  0.544406 0.039297
           cash-flow + anoma

## Did the fourth signal rescue the fusion?

In [3]:
full4  = ab.loc[ab["signals"]==" + ".join(LABEL[c] for c in SIGNALS), "auc_mean"].iloc[0]
surv   = ab.loc[ab["signals"]=="survival", "auc_mean"].iloc[0]
best   = ab.iloc[0]
three  = ab.loc[ab["signals"]=="survival + cash-flow + anomaly", "auc_mean"]
three  = three.iloc[0] if len(three) else float("nan")

print(f"survival alone                : {surv:.3f}")
print(f"three-signal (nb 09 repeat)   : {three:.3f}")
print(f"four-signal fusion            : {full4:.3f}")
print(f"best subset overall           : {best['signals']}  ({best['auc_mean']:.3f})")
print()
if full4 > surv:
    print(f"VERDICT: the four-signal fusion beats survival alone by {full4-surv:+.3f} AUC.")
    print("The notebook-09 hypothesis is SUPPORTED -- the distress-language signal was")
    print("carrying the complementary information.")
else:
    print(f"VERDICT: the four-signal fusion still does NOT beat survival alone ({full4-surv:+.3f} AUC).")
    print("The notebook-09 hypothesis is REFUTED. Report it that way. Candidate reasons to")
    print("state: the text signal is undated company-type language rather than temporal")
    print("distress (see notebook 11); the cash-flow signal still rests on placeholder")
    print("starting cash; or the signals are genuinely redundant with the hazard estimate.")

survival alone                : 0.753
three-signal (nb 09 repeat)   : 0.616
four-signal fusion            : 0.544
best subset overall           : survival  (0.753)

VERDICT: the four-signal fusion still does NOT beat survival alone (-0.209 AUC).
The notebook-09 hypothesis is REFUTED. Report it that way. Candidate reasons to
state: the text signal is undated company-type language rather than temporal
distress (see notebook 11); the cash-flow signal still rests on placeholder
starting cash; or the signals are genuinely redundant with the hazard estimate.


In [5]:
fusion = LogisticRegression(max_iter=1000).fit(m[SIGNALS].fillna(0), y)
m["RHI"] = (100*(1 - fusion.predict_proba(m[SIGNALS].fillna(0))[:,1])).round(1)
weights = dict(zip([LABEL[c] for c in SIGNALS], np.round(fusion.coef_[0],4)))
print("[weights fitted on full data]", weights)
print(f"[RHI] median {m['RHI'].median():.1f} across {len(m):,} scored companies")

pd.DataFrame([{
    "signals_used":"survival + cash-flow + anomaly + distress-language",
    "companies": len(m),
    "auc_4signal": round(full4,4),
    "auc_survival_alone": round(surv,4),
    "auc_3signal_prev": round(three,4) if three==three else None,
    "best_subset": best["signals"], "best_auc": round(best["auc_mean"],4),
    "verdict": "fusion beats single" if full4>surv else "fusion does not beat single",
}]).to_csv(f"{REPORTS}/experiment5_fusion_full.csv", index=False)
m[["id","RHI"]].to_csv(f"{PROCESSED}/rhi_scores.csv", index=False)
print(f"[done] wrote {REPORTS}/experiment5_fusion_full.csv and {PROCESSED}/rhi_scores.csv")

[weights fitted on full data] {'survival': np.float64(-0.2085), 'cash-flow': np.float64(0.0632), 'anomaly': np.float64(1.4202), 'distress-language': np.float64(0.7205)}
[RHI] median 56.8 across 1,892 scored companies
[done] wrote ../reports/experiment5_fusion_full.csv and ../data/processed/rhi_scores.csv
